# Medical Transcription Classification — Complete Notebook

## Section 0: Project Overview

### Goal
Classify medical transcriptions into **4 categories** using Natural Language Processing and Machine Learning.

### Dataset Description
Each row contains three columns:
- **label** — integer 1–4 indicating the medical specialty class
- **description** — short title/procedure name
- **text** — full transcription of the medical record

### Classes
| Label | Class |
|-------|-------|
| 1 | Surgery |
| 2 | Medical Records |
| 3 | Internal Medicine |
| 4 | Other |

### Pipeline
This notebook follows a structured end-to-end pipeline:
1. **EDA** — Explore the data, understand class distributions, text lengths, and vocabulary.
2. **Preprocessing** — Clean text, tokenize using a SNOMED CT vocabulary, and pad sequences.
3. **RNN/LSTM Classifier** — Train a deep-learning sequential model on padded token sequences.
4. **Baseline Models** — Train TF-IDF + Logistic Regression, SVM, and Random Forest classifiers.
5. **Comparison** — Compare all models on accuracy, F1-score, training time, and interpretability.

## Section 1: Setup & Imports

Import all required libraries. `wordcloud` is optional — the notebook degrades gracefully if it is not installed.

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import re, time, collections
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

try:
    from wordcloud import WordCloud
    HAS_WORDCLOUD = True
except ImportError:
    HAS_WORDCLOUD = False
    print("wordcloud not installed — word cloud cells will be skipped")

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dropout, Dense
from tensorflow.keras.preprocessing.sequence import pad_sequences

sns.set_theme(style='whitegrid', palette='tab10')
PALETTE = sns.color_palette('tab10', 4)
plt.rcParams.update({'figure.dpi': 100, 'axes.titlesize': 13, 'axes.labelsize': 11})

BASE          = Path('.')
TRAIN_CSV     = BASE / 'train.csv'
TEST_CSV      = BASE / 'test.csv'
CLASSES_TXT   = BASE / 'classes.txt'
STOPWORDS_TXT = BASE / 'clinical-stopwords.txt'
VOCAB_TXT     = BASE / 'vocab.txt'

print(f"TensorFlow version: {tf.__version__}")

## Section 2: Load Auxiliary Files

Three auxiliary files are provided alongside the dataset:
- **`classes.txt`** — one class name per line (lines 1–4 map to labels 1–4).
- **`clinical-stopwords.txt`** — clinical stop words to remove during tokenization.
  Standard stop-word lists miss domain-specific noise words like *patient*, *denies*, *history*.
- **`vocab.txt`** — SNOMED CT–derived vocabulary that ensures professional medical terminology
  is recognised during tokenisation and model training.

In [ ]:
# Load class names from classes.txt
CLASS_NAMES = {}
with open(CLASSES_TXT) as f:
    for idx, line in enumerate(f, start=1):
        name = line.strip()
        if name:
            CLASS_NAMES[idx] = name
print("Classes:", CLASS_NAMES)

# Load clinical stop words
STOPWORDS = set()
with open(STOPWORDS_TXT) as f:
    for line in f:
        w = line.strip()
        if w and not w.startswith('#'):
            STOPWORDS.add(w.lower())
print(f"Loaded {len(STOPWORDS):,} clinical stop-words")

# Load SNOMED CT vocabulary
VOCAB = []
with open(VOCAB_TXT) as f:
    for line in f:
        w = line.strip()
        if w:
            VOCAB.append(w)
print(f"Loaded {len(VOCAB):,} vocabulary terms")
print(f"Sample vocab terms: {VOCAB[:10]}")

## Section 3: Load & Explore Data

Load the pre-split train and test CSV files. We fill any missing text values with empty strings,
map numeric labels to human-readable class names, and compute character and word-count features
that will be used throughout the EDA.

In [ ]:
train_df = pd.read_csv(TRAIN_CSV)
test_df  = pd.read_csv(TEST_CSV)

# Fill missing text values
train_df['text'] = train_df['text'].fillna('')
test_df['text']  = test_df['text'].fillna('')

# Map labels to class names
train_df['class'] = train_df['label'].map(CLASS_NAMES)
test_df['class']  = test_df['label'].map(CLASS_NAMES)

# Add length features
train_df['char_len'] = train_df['text'].str.len()
train_df['word_len'] = train_df['text'].str.split().str.len()
test_df['char_len']  = test_df['text'].str.len()
test_df['word_len']  = test_df['text'].str.split().str.len()

print(f"Train shape: {train_df.shape}")
print(f"Test shape:  {test_df.shape}")
train_df.head()

## Section 4: Exploratory Data Analysis (EDA)

This section systematically explores the dataset across ten dimensions:
class distribution, text length, word frequency, word clouds, n-gram analysis,
vocabulary coverage, token length CDF, TF-IDF correlation, train/test split, and missing data.

### 4.1 Class Distribution

Visualise how samples are spread across the four medical specialties.

In [ ]:
counts = train_df['class'].value_counts().reindex(CLASS_NAMES.values())
pcts   = counts / counts.sum() * 100

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
bars = axes[0].bar(counts.index, counts.values, color=PALETTE)
axes[0].set_title('Class Distribution (Train Set)')
axes[0].set_xlabel('Medical Specialty')
axes[0].set_ylabel('Number of Samples')
axes[0].tick_params(axis='x', rotation=15)
for bar, cnt, pct in zip(bars, counts.values, pcts.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 10,
                 f'{cnt}\n({pct:.1f}%)', ha='center', va='bottom', fontsize=9)
axes[1].pie(counts.values, labels=counts.index, autopct='%1.1f%%',
            colors=PALETTE, startangle=140,
            wedgeprops={'edgecolor': 'white', 'linewidth': 1.5})
axes[1].set_title('Class Distribution — Pie Chart (Train Set)')
plt.tight_layout(); plt.show()

### 4.2 Text Length Analysis

Examine how text length (characters and words) varies overall and per class.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes[0,0].hist(train_df['char_len'], bins=50, color='steelblue', edgecolor='white')
axes[0,0].set_title('Character Count Distribution')
axes[0,0].set_xlabel('Characters per Sample'); axes[0,0].set_ylabel('Frequency')
axes[0,1].hist(train_df['word_len'], bins=50, color='darkorange', edgecolor='white')
axes[0,1].set_title('Word Count Distribution')
axes[0,1].set_xlabel('Words per Sample'); axes[0,1].set_ylabel('Frequency')
order = list(CLASS_NAMES.values())
sns.boxplot(data=train_df, x='class', y='word_len', order=order, palette=PALETTE, ax=axes[1,0])
axes[1,0].set_title('Word Count by Class (Box Plot)')
axes[1,0].set_xlabel('Class'); axes[1,0].set_ylabel('Word Count')
axes[1,0].tick_params(axis='x', rotation=15)
sns.violinplot(data=train_df, x='class', y='word_len', order=order, palette=PALETTE, ax=axes[1,1], inner='quartile')
axes[1,1].set_title('Word Count by Class (Violin Plot)')
axes[1,1].set_xlabel('Class'); axes[1,1].set_ylabel('Word Count')
axes[1,1].tick_params(axis='x', rotation=15)
plt.tight_layout(); plt.show()

### 4.3 Word Frequency Analysis

Identify the most common meaningful words overall and within each class, after removing clinical stop words.

In [ ]:
def tokenize(text, stopwords=STOPWORDS):
    """Lowercase, keep only alphabetic tokens, remove stopwords."""
    tokens = re.findall(r'[a-z]+', str(text).lower())
    return [t for t in tokens if t not in stopwords and len(t) > 2]

TOP_N = 20
all_tokens = [t for text in train_df['text'] for t in tokenize(text)]
overall_freq = collections.Counter(all_tokens).most_common(TOP_N)
words, freqs = zip(*overall_freq)

fig, ax = plt.subplots(figsize=(12, 5))
ax.barh(words[::-1], freqs[::-1], color='steelblue')
ax.set_title(f'Top {TOP_N} Most Frequent Words (Overall)')
ax.set_xlabel('Frequency'); ax.set_ylabel('Word')
plt.tight_layout(); plt.show()

fig, axes = plt.subplots(2, 2, figsize=(16, 12))
for ax, (label, cls_name) in zip(axes.flat, CLASS_NAMES.items()):
    cls_texts = train_df[train_df['label'] == label]['text']
    tokens = [t for text in cls_texts for t in tokenize(text)]
    top = collections.Counter(tokens).most_common(15)
    if top:
        ws, fs = zip(*top)
        ax.barh(ws[::-1], fs[::-1], color=PALETTE[label-1])
    ax.set_title(f'Top 15 Words — {cls_name}')
    ax.set_xlabel('Frequency')
plt.suptitle('Top 15 Most Frequent Words per Class', fontsize=14, y=1.01)
plt.tight_layout(); plt.show()

### 4.4 Word Clouds

Visual representation of word frequency — larger words appear more often in the transcriptions.

In [ ]:
if HAS_WORDCLOUD:
    wc_kwargs = dict(background_color='white', max_words=200,
                     stopwords=STOPWORDS, width=800, height=400, colormap='viridis')
    all_text = ' '.join(train_df['text'].fillna(''))
    wc = WordCloud(**wc_kwargs).generate(all_text)
    fig, ax = plt.subplots(figsize=(14, 6))
    ax.imshow(wc, interpolation='bilinear'); ax.axis('off')
    ax.set_title('Overall Word Cloud — Medical Transcriptions', fontsize=14)
    plt.tight_layout(); plt.show()

    colormaps = ['Blues', 'Oranges', 'Greens', 'Purples']
    fig, axes = plt.subplots(2, 2, figsize=(16, 10))
    for ax, (label, cls_name), cmap in zip(axes.flat, CLASS_NAMES.items(), colormaps):
        cls_text = ' '.join(train_df[train_df['label'] == label]['text'].fillna(''))
        wc = WordCloud(**{**wc_kwargs, 'colormap': cmap}).generate(cls_text)
        ax.imshow(wc, interpolation='bilinear'); ax.axis('off')
        ax.set_title(cls_name, fontsize=12)
    plt.suptitle('Per-Class Word Clouds', fontsize=14)
    plt.tight_layout(); plt.show()
else:
    print("Skipping word clouds — install wordcloud: pip install wordcloud")

### 4.5 N-gram Analysis

Bigrams and trigrams reveal common multi-word clinical phrases that single-word counts miss.

In [ ]:
def get_ngrams(texts, n=2, top_k=15, stopwords=STOPWORDS):
    ngrams = []
    for text in texts:
        tokens = tokenize(text, stopwords)
        ngrams.extend(zip(*[tokens[i:] for i in range(n)]))
    return collections.Counter(ngrams).most_common(top_k)

def plot_ngrams(ngram_counts, title, ax, color):
    if not ngram_counts: return
    labels = [' '.join(g) for g, _ in ngram_counts]
    vals   = [c for _, c in ngram_counts]
    ax.barh(labels[::-1], vals[::-1], color=color)
    ax.set_title(title); ax.set_xlabel('Count')

bigrams  = get_ngrams(train_df['text'], n=2)
trigrams = get_ngrams(train_df['text'], n=3)
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
plot_ngrams(bigrams,  'Top 15 Bigrams (Overall)',  axes[0], 'steelblue')
plot_ngrams(trigrams, 'Top 15 Trigrams (Overall)', axes[1], 'darkorange')
plt.tight_layout(); plt.show()

fig, axes = plt.subplots(2, 2, figsize=(16, 12))
for ax, (label, cls_name) in zip(axes.flat, CLASS_NAMES.items()):
    cls_texts = train_df[train_df['label'] == label]['text']
    bg = get_ngrams(cls_texts, n=2, top_k=10)
    plot_ngrams(bg, f'Top 10 Bigrams — {cls_name}', ax, PALETTE[label-1])
plt.suptitle('Top Bigrams per Class', fontsize=14, y=1.01)
plt.tight_layout(); plt.show()

### 4.6 Vocabulary Coverage

Compare the total and unique token counts per class, and the type-token ratio as a measure of lexical diversity.

In [ ]:
vocab_data = []
for label, cls_name in CLASS_NAMES.items():
    cls_texts = train_df[train_df['label'] == label]['text']
    tokens = [t for text in cls_texts for t in tokenize(text)]
    vocab_data.append({'Class': cls_name, 'Total Tokens': len(tokens),
                       'Unique Tokens': len(set(tokens)),
                       'Type-Token Ratio': round(len(set(tokens)) / max(len(tokens), 1), 4)})
vocab_df = pd.DataFrame(vocab_data)
print(vocab_df.to_string(index=False))

x = np.arange(len(vocab_df)); width = 0.35
fig, ax = plt.subplots(figsize=(11, 5))
ax.bar(x - width/2, vocab_df['Total Tokens'],  width, label='Total Tokens',  color='steelblue')
ax.bar(x + width/2, vocab_df['Unique Tokens'], width, label='Unique Tokens', color='darkorange')
ax.set_xticks(x); ax.set_xticklabels(vocab_df['Class'], rotation=10)
ax.set_title('Vocabulary Coverage per Class'); ax.set_ylabel('Token Count'); ax.legend()
plt.tight_layout(); plt.show()

### 4.7 Token Length Distribution & CDF

The CDF helps us choose the padding length `MAX_LEN` so that the vast majority of samples are fully represented.

In [ ]:
token_counts = train_df['text'].apply(lambda t: len(tokenize(t)))
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].hist(token_counts, bins=50, color='teal', edgecolor='white')
axes[0].set_title('Token Count Distribution per Sample')
axes[0].set_xlabel('Number of Tokens (after stopword removal)'); axes[0].set_ylabel('Frequency')
sorted_tc = np.sort(token_counts)
cdf = np.arange(1, len(sorted_tc)+1) / len(sorted_tc)
axes[1].plot(sorted_tc, cdf, color='teal', linewidth=2)
axes[1].axhline(0.90, color='red',    linestyle='--', label='90th percentile')
axes[1].axhline(0.95, color='orange', linestyle='--', label='95th percentile')
axes[1].set_title('CDF of Token Length'); axes[1].set_xlabel('Token Count')
axes[1].set_ylabel('Cumulative Proportion'); axes[1].legend()
plt.tight_layout(); plt.show()
print(f"Median tokens: {token_counts.median():.0f}")
print(f"90th percentile: {np.percentile(token_counts, 90):.0f}")
print(f"95th percentile: {np.percentile(token_counts, 95):.0f}")

### 4.8 TF-IDF Feature Correlation Heatmap

Shows pairwise correlation between the top 30 TF-IDF terms, revealing clusters of co-occurring clinical language.

In [ ]:
vectorizer_eda = TfidfVectorizer(max_features=30, stop_words=list(STOPWORDS),
                                  token_pattern=r'(?u)\b[a-zA-Z]{3,}\b')
tfidf_eda = vectorizer_eda.fit_transform(train_df['text'].fillna(''))
tfidf_eda_df = pd.DataFrame(tfidf_eda.toarray(), columns=vectorizer_eda.get_feature_names_out())
corr = tfidf_eda_df.corr()
fig, ax = plt.subplots(figsize=(14, 11))
sns.heatmap(corr, ax=ax, cmap='RdBu_r', center=0, square=True, linewidths=0.5,
            annot=False, cbar_kws={'shrink': 0.8})
ax.set_title('TF-IDF Feature Correlation Heatmap (Top 30 Terms)', fontsize=13)
plt.xticks(rotation=45, ha='right'); plt.yticks(rotation=0)
plt.tight_layout(); plt.show()

### 4.9 Train/Test Split Analysis

Verify that the class distribution is consistent between training and test sets.

In [ ]:
train_counts = train_df['class'].value_counts().reindex(CLASS_NAMES.values(), fill_value=0)
test_counts  = test_df['class'].value_counts().reindex(CLASS_NAMES.values(), fill_value=0)
train_pct = train_counts / train_counts.sum() * 100
test_pct  = test_counts  / test_counts.sum()  * 100

x = np.arange(len(CLASS_NAMES)); width = 0.35
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].bar(x-width/2, train_counts.values, width, label='Train', color='steelblue')
axes[0].bar(x+width/2, test_counts.values,  width, label='Test',  color='darkorange')
axes[0].set_xticks(x); axes[0].set_xticklabels(list(CLASS_NAMES.values()), rotation=10)
axes[0].set_title('Class Count — Train vs. Test'); axes[0].set_ylabel('Samples'); axes[0].legend()
axes[1].bar(x-width/2, train_pct.values, width, label='Train', color='steelblue')
axes[1].bar(x+width/2, test_pct.values,  width, label='Test',  color='darkorange')
axes[1].set_xticks(x); axes[1].set_xticklabels(list(CLASS_NAMES.values()), rotation=10)
axes[1].set_title('Class Proportion (%) — Train vs. Test'); axes[1].set_ylabel('Percentage (%)'); axes[1].legend()
plt.tight_layout(); plt.show()

### 4.10 Missing Data Summary

Check for any missing values in the key columns before modelling.

In [ ]:
def missing_summary(df, name):
    ms = pd.DataFrame({'Missing Count': df.isnull().sum(),
                       'Missing %': (df.isnull().sum()/len(df)*100).round(2),
                       'Dtype': df.dtypes})
    print(f"\n{'='*40}\nMissing data — {name}\n{'='*40}")
    print(ms.to_string())
    return ms

train_missing = missing_summary(train_df[['label','description','text']], 'Train')
test_missing  = missing_summary(test_df[['label','description','text']],  'Test')

fig, axes = plt.subplots(1, 2, figsize=(10, 3))
for ax, df, title in [(axes[0], train_df[['label','description','text']], 'Train'),
                      (axes[1], test_df[['label','description','text']],  'Test')]:
    sns.heatmap(df.isnull(), ax=ax, yticklabels=False, cbar=False, cmap='OrRd')
    ax.set_title(f'Missing Values Heatmap — {title}')
plt.tight_layout(); plt.show()

## Section 5: Data Preprocessing for Models

Before training the LSTM, we convert raw text into padded integer sequences:
1. **Vocabulary mapping** — build a `word2idx` dictionary from `vocab.txt`.
   Words not found in the vocabulary are mapped to index 1 (`<UNK>`).
2. **Cleaning** — lowercase text, strip non-alphabetic characters, remove clinical stop words.
3. **Encoding** — replace each token with its integer index.
4. **Padding / truncating** — all sequences are padded (or truncated) to `MAX_LEN = 300` tokens.
   The EDA CDF shows this covers the great majority of samples.

In [ ]:
# Build word-to-index mapping from vocab.txt
word2idx = {'<PAD>': 0, '<UNK>': 1}
for word in VOCAB:
    if word not in word2idx:
        word2idx[word] = len(word2idx)
VOCAB_SIZE = len(word2idx)
MAX_LEN = 300
print(f"Vocabulary size (with PAD+UNK): {VOCAB_SIZE:,}")
print(f"Sequence length: {MAX_LEN}")

def clean_and_tokenize(text):
    """Lowercase, remove non-alpha, remove stop words, split into tokens."""
    text = str(text).lower()
    text = re.sub(r'[^a-z\s]', ' ', text)
    tokens = text.split()
    return [t for t in tokens if t not in STOPWORDS and len(t) > 1]

def encode(tokens):
    """Map tokens to indices; unknown words → index 1 (<UNK>)."""
    return [word2idx.get(t, 1) for t in tokens]

print("\nPreprocessing training data (this may take a moment)...")
X_train_tokens = train_df['text'].apply(clean_and_tokenize)
X_test_tokens  = test_df['text'].apply(clean_and_tokenize)

X_train_idx = X_train_tokens.apply(encode).tolist()
X_test_idx  = X_test_tokens.apply(encode).tolist()

X_train_pad = pad_sequences(X_train_idx, maxlen=MAX_LEN, padding='post', truncating='post')
X_test_pad  = pad_sequences(X_test_idx,  maxlen=MAX_LEN, padding='post', truncating='post')

# Labels: convert 1-4 → 0-3
y_train = (train_df['label'].values - 1).astype(int)
y_test  = (test_df['label'].values  - 1).astype(int)

print(f"\nX_train_pad shape : {X_train_pad.shape}")
print(f"X_test_pad  shape : {X_test_pad.shape}")
print(f"y_train unique labels: {np.unique(y_train)}")

## Section 6: Part 1 — RNN/LSTM Classifier

We train a four-layer sequential LSTM network:
- **Embedding Layer** — learns dense 128-dimensional vector representations for each word index.
- **LSTM Layer (128 units)** — captures long-range sequential dependencies in the transcription text.
  LSTMs are preferred over vanilla RNNs because the gating mechanism mitigates the vanishing-gradient
  problem that arises with long medical reports.
- **Dropout (0.3)** — regularises the network to reduce overfitting on the limited training set.
- **Dense Softmax (4 units)** — outputs a probability distribution over the four classes.

The model is compiled with the Adam optimiser and sparse categorical cross-entropy loss (labels are integers, not one-hot).

In [ ]:
# Build LSTM model
model = Sequential([
    Embedding(input_dim=VOCAB_SIZE, output_dim=128, input_length=MAX_LEN),
    LSTM(128),
    Dropout(0.3),
    Dense(4, activation='softmax')
], name='LSTM_Classifier')

model.compile(optimizer='adam',
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])
model.summary()

# Train
print("\nTraining LSTM model (10 epochs)...")
rnn_start = time.time()
history = model.fit(
    X_train_pad, y_train,
    epochs=10, batch_size=32,
    validation_split=0.1,
    verbose=1
)
rnn_time = time.time() - rnn_start
print(f"\nTotal training time: {rnn_time:.1f}s")

# Training curves
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(history.history['accuracy'],     label='Train Acc')
axes[0].plot(history.history['val_accuracy'], label='Val Acc')
axes[0].set_title('LSTM Accuracy'); axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Accuracy'); axes[0].legend()
axes[1].plot(history.history['loss'],     label='Train Loss')
axes[1].plot(history.history['val_loss'], label='Val Loss')
axes[1].set_title('LSTM Loss'); axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Loss'); axes[1].legend()
plt.tight_layout(); plt.show()

# Evaluate
rnn_loss, rnn_acc = model.evaluate(X_test_pad, y_test, verbose=0)
rnn_preds = np.argmax(model.predict(X_test_pad, verbose=0), axis=1)
rnn_f1 = f1_score(y_test, rnn_preds, average='macro')

CLASS_LABELS = ['Surgery', 'Medical Records', 'Internal Medicine', 'Other']

print(f"\nRNN Test Accuracy : {rnn_acc:.4f}")
print(f"RNN Macro F1-Score: {rnn_f1:.4f}")
print("\nClassification Report:")
print(classification_report(y_test, rnn_preds, target_names=CLASS_LABELS))

# Confusion matrix
cm = confusion_matrix(y_test, rnn_preds)
fig, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=CLASS_LABELS, yticklabels=CLASS_LABELS, ax=ax)
ax.set_title('RNN/LSTM Confusion Matrix')
ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')
plt.tight_layout(); plt.show()

## Section 7: Part 2 — Baseline Models (TF-IDF)

We evaluate three classical machine-learning classifiers using **TF-IDF** features:
- **Logistic Regression** — often the strongest simple baseline for text; fast and fully interpretable
  (coefficients reveal which words drive each prediction).
- **LinearSVC** — excellent at high-dimensional sparse feature spaces typical of TF-IDF vectors.
- **Random Forest** — captures non-linear interactions between specific medical keywords.

TF-IDF does not model word order, so these models are less suited to capturing long-range dependencies
than the LSTM — but they are much faster to train and often competitive on short texts.

In [ ]:
results = {}

# TF-IDF
print("Fitting TF-IDF vectorizer (max_features=10000)...")
tfidf = TfidfVectorizer(max_features=10000, stop_words=list(STOPWORDS),
                        token_pattern=r'(?u)\b[a-zA-Z]{2,}\b')
X_train_tfidf = tfidf.fit_transform(train_df['text'])
X_test_tfidf  = tfidf.transform(test_df['text'])
print(f"TF-IDF matrix shape: {X_train_tfidf.shape}")

# Logistic Regression
print("\n--- Logistic Regression ---")
t0 = time.time()
lr = LogisticRegression(max_iter=1000, random_state=42)
lr.fit(X_train_tfidf, y_train)
lr_time = time.time() - t0
lr_preds = lr.predict(X_test_tfidf)
lr_acc = accuracy_score(y_test, lr_preds)
lr_f1  = f1_score(y_test, lr_preds, average='macro')
results['Logistic Regression'] = {'Accuracy': round(lr_acc,4), 'F1 (Macro)': round(lr_f1,4),
                                   'Time (s)': round(lr_time,2), 'Interpretability': 'High'}
print(f"Accuracy: {lr_acc:.4f}  |  Macro F1: {lr_f1:.4f}  |  Time: {lr_time:.2f}s")
print(classification_report(y_test, lr_preds, target_names=CLASS_LABELS))

# SVM
print("\n--- Support Vector Machine (LinearSVC) ---")
t0 = time.time()
svm = LinearSVC(max_iter=2000, random_state=42)
svm.fit(X_train_tfidf, y_train)
svm_time = time.time() - t0
svm_preds = svm.predict(X_test_tfidf)
svm_acc = accuracy_score(y_test, svm_preds)
svm_f1  = f1_score(y_test, svm_preds, average='macro')
results['SVM'] = {'Accuracy': round(svm_acc,4), 'F1 (Macro)': round(svm_f1,4),
                  'Time (s)': round(svm_time,2), 'Interpretability': 'Medium'}
print(f"Accuracy: {svm_acc:.4f}  |  Macro F1: {svm_f1:.4f}  |  Time: {svm_time:.2f}s")
print(classification_report(y_test, svm_preds, target_names=CLASS_LABELS))

# Random Forest
print("\n--- Random Forest ---")
t0 = time.time()
rf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf.fit(X_train_tfidf, y_train)
rf_time = time.time() - t0
rf_preds = rf.predict(X_test_tfidf)
rf_acc = accuracy_score(y_test, rf_preds)
rf_f1  = f1_score(y_test, rf_preds, average='macro')
results['Random Forest'] = {'Accuracy': round(rf_acc,4), 'F1 (Macro)': round(rf_f1,4),
                             'Time (s)': round(rf_time,2), 'Interpretability': 'Medium'}
print(f"Accuracy: {rf_acc:.4f}  |  Macro F1: {rf_f1:.4f}  |  Time: {rf_time:.2f}s")
print(classification_report(y_test, rf_preds, target_names=CLASS_LABELS))

## Section 8: Model Comparison

We compare all four models side-by-side across four dimensions:
- **Accuracy** — proportion of correctly classified test samples.
- **F1 (Macro)** — unweighted average F1 over all classes; punishes poor performance on minority classes.
- **Training Time** — wall-clock seconds; important for deployment and iteration speed.
- **Interpretability** — whether clinicians can trace which features drove a prediction.

In [ ]:
# Add RNN results
results['RNN (LSTM)'] = {'Accuracy': round(rnn_acc,4), 'F1 (Macro)': round(rnn_f1,4),
                          'Time (s)': round(rnn_time,2), 'Interpretability': 'Low'}

comparison_df = (pd.DataFrame(results).T
                   .reset_index()
                   .rename(columns={'index': 'Model'}))
comparison_df['Accuracy']   = comparison_df['Accuracy'].astype(float)
comparison_df['F1 (Macro)'] = comparison_df['F1 (Macro)'].astype(float)
comparison_df = comparison_df.sort_values('Accuracy', ascending=False).reset_index(drop=True)

print("\n=== MODEL COMPARISON ===")
print(comparison_df.to_string(index=False))

# Accuracy bar chart
fig, ax = plt.subplots(figsize=(10, 5))
colors = sns.color_palette('tab10', len(comparison_df))
bars = ax.bar(comparison_df['Model'], comparison_df['Accuracy'], color=colors)
ax.set_title('Model Accuracy Comparison')
ax.set_ylabel('Accuracy'); ax.set_ylim(0, 1.05)
for bar, acc in zip(bars, comparison_df['Accuracy']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{acc:.4f}', ha='center', va='bottom', fontsize=10)
plt.tight_layout(); plt.show()

# F1 bar chart
fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(comparison_df['Model'], comparison_df['F1 (Macro)'], color=colors)
ax.set_title('Model Macro F1-Score Comparison')
ax.set_ylabel('Macro F1-Score'); ax.set_ylim(0, 1.05)
for bar, f1 in zip(bars, comparison_df['F1 (Macro)']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{f1:.4f}', ha='center', va='bottom', fontsize=10)
plt.tight_layout(); plt.show()

comparison_df.to_csv('comparison_results.csv', index=False)
print("Results saved to comparison_results.csv")

## Section 9: Conclusion

### Key Findings
- **Best model** — The comparison table above identifies which model achieved the highest accuracy and
  macro F1 on the held-out test set. In practice, LinearSVC and Logistic Regression are often competitive
  with the LSTM on datasets of this size (~4,500 training samples).

### Trade-offs
| Model | Strength | Weakness |
|-------|----------|----------|
| RNN (LSTM) | Captures sequential dependencies; scales with data | Slow to train; black-box |
| Logistic Regression | Fast; highly interpretable; strong baseline | Ignores word order |
| SVM | Strong high-dim performance; moderate interpretability | Harder to tune |
| Random Forest | Non-linear keyword interactions | Slow, lower accuracy on text |

### Interpretability in Medical NLP
In clinical decision-support systems, a model that achieves 2 % higher accuracy but cannot explain
its predictions may be **less useful** than a Logistic Regression whose top positive coefficients
show, for example, that *incised* or *suture* strongly predict **Surgery**, or that *palpitation*
or *dyspnoea* predict **Internal Medicine**. Clinicians can verify and trust such explanations.

### Potential Improvements
1. **Pre-trained clinical embeddings** — BioWordVec or ClinicalBERT embeddings instead of random
   initialisation would give the LSTM a significant head start.
2. **Transformer models** — Fine-tuning BioBERT or ClinicalBERT would likely outperform all models here.
3. **Data augmentation** — Back-translation or synonym replacement could address class imbalance.
4. **Hyperparameter search** — Grid / random search on LSTM depth, hidden size, learning rate.

### Limitations
- **Small dataset** (~4,500 training samples) limits deep-learning headroom.
- **Class imbalance** — Surgery is over-represented; macro F1 penalises this.
- **Simple tokenisation** — vocabulary-based tokenisation misses sub-word morphology.
- **No cross-validation** — a single train/test split may not reflect true generalisation.